In [2]:
import sys
import os
sys.path.append(os.path.abspath('..'))

In [3]:
import sys, os
sys.path.append(os.path.abspath('..'))

from src.config import TRAIN_CSV, TRAIN_IMGS
print(TRAIN_CSV)
print(TRAIN_IMGS)

e:\Portfolio\Medical_Imaging_Diagnosis_API\data\train_1.csv
e:\Portfolio\Medical_Imaging_Diagnosis_API\data\train_images\train_images


In [18]:
from src.dataset import RetinalDataset, train_transforms, val_transforms
from src.config import TRAIN_CSV, VAL_CSV, TRAIN_IMGS, VAL_IMGS, BATCH_SIZE
from torch.utils.data import DataLoader

train_dataset = RetinalDataset(TRAIN_CSV, TRAIN_IMGS, transform=train_transforms)
val_dataset   = RetinalDataset(VAL_CSV,   VAL_IMGS,   transform=val_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

images, labels = next(iter(train_loader))
print("Batch images shape:", images.shape)
print("Batch labels shape:", labels.shape)
print("Labels in this batch:", labels)


Batch images shape: torch.Size([32, 3, 224, 224])
Batch labels shape: torch.Size([32])
Labels in this batch: tensor([1, 1, 0, 4, 2, 1, 1, 0, 2, 0, 2, 0, 1, 2, 2, 0, 0, 2, 2, 0, 0, 0, 0, 0,
        2, 0, 2, 2, 2, 2, 1, 0])


In [13]:
from src.dataset import RetinalDataset, train_transforms

dataset = RetinalDataset("../data/train_1.csv", "../data/train_images/train_images", transform=train_transforms)
print("Dataset size:", len(dataset))

image, label = dataset[0]
print("Image tensor shape:", image.shape)
print("Label:", label)

Dataset size: 2930
Image tensor shape: torch.Size([3, 224, 224])
Label: 2


In [20]:
from src.model import build_model
import torch

model = build_model(num_classes=5, pretrained=True)

dummy_input = torch.randn(32, 3, 224, 224)
output = model(dummy_input)

print("Output shape:", output.shape)
print("Output sample:", output[0])

Downloading: "https://download.pytorch.org/models/efficientnet_b4_rwightman-23ab8bcd.pth" to C:\Users\la_re/.cache\torch\hub\checkpoints\efficientnet_b4_rwightman-23ab8bcd.pth


100%|██████████| 74.5M/74.5M [02:24<00:00, 541kB/s]


Output shape: torch.Size([32, 5])
Output sample: tensor([ 0.0425,  0.0197, -0.1156, -0.0082,  0.0642],
       grad_fn=<SelectBackward0>)


In [21]:
from src.train import train

train(num_epochs=2)

Training on: cpu


e:\Portfolio\Medical_Imaging_Diagnosis_API\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
e:\Portfolio\Medical_Imaging_Diagnosis_API\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B4_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B4_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


\nEpoch 1/2
----------------------------------------
  Batch 0/92 - Loss: 1.6327
  Batch 10/92 - Loss: 1.5776
  Batch 20/92 - Loss: 1.5829


KeyboardInterrupt: 

In [ ]:
from src.predict import load_model, predict
from PIL import Image
from src.config import VAL_IMGS
import pandas as pd
from src.config import VAL_CSV

# Load model
model, device = load_model()

# Pick a real image from val set
val_df = pd.read_csv(VAL_CSV)
sample = val_df.iloc[0]
image  = Image.open(f"{VAL_IMGS}/{sample['id_code']}.png").convert('RGB')

# Predict
result = predict(image, model, device)
print(f"True label:  {sample['diagnosis']}")
print(f"Predicted:   {result['predicted_class']} - {result['class_name']}")
print(f"Confidence:  {result['confidence']*100:.1f}%")
print(f"\\nAll probabilities:")
for cls, prob in result['probabilities'].items():
    print(f"  {cls}: {prob*100:.1f}%")

Pretrained weights loaded successfully
Model loaded from e:\Portfolio\Medical_Imaging_Diagnosis_API\models\best_model.pth on cpu
True label:  2
Predicted:   2 - Moderate DR
Confidence:  68.2%
\nAll probabilities:
  No Diabetic Retinopathy: 0.0%
  Mild DR: 5.0%
  Moderate DR: 68.2%
  Severe DR: 16.4%
  Proliferative DR: 10.4%


: 